# ResNet‑18 Image Embedding and Feature‑Map Interpretation

This notebook explains **how a ResNet‑18 model transforms an image step‑by‑step** using convolutional layers with **3×3 kernels and stride 1** (inside residual blocks).

We explicitly explain:
- what feature maps represent at each layer
- how visual abstraction evolves
- how the final 512‑D embedding is formed


## 1. Imports
Libraries for deep learning, visualization, and numerical processing.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from matplotlib.patches import FancyArrowPatch


## 2. Device selection

CPU is used for Binder and cross‑platform compatibility.

In [ ]:
device = torch.device("cpu")
print("Using device:", device)

## 3. Load ResNet‑18 in embedding mode

The final classification layer is removed. The network now outputs a **512‑dimensional embedding**, not class scores.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval()
print("ResNet‑18 loaded in embedding mode")

## 4. Feature‑map hooks (store intermediate representations)

Feature maps represent **responses of learned filters**.
Each channel answers the question:

> *Where in the image does my learned pattern occur, and how strongly?*

In [ ]:
feature_maps = {}

def hook_fn(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

model.conv1.register_forward_hook(hook_fn("conv1"))
model.layer1.register_forward_hook(hook_fn("layer1"))
model.layer2.register_forward_hook(hook_fn("layer2"))
model.layer3.register_forward_hook(hook_fn("layer3"))
model.layer4.register_forward_hook(hook_fn("layer4"))


## 5. Image preprocessing

The image is resized and center‑cropped to conform to ImageNet training statistics.
No padding is used.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

## 6. Load input image

In [ ]:
img_path = Path("data") / "25DIV3FAY_1002.jpg"
assert img_path.exists(), f"Missing image: {img_path}"

img = Image.open(img_path).convert("RGB")
x = transform(img).unsqueeze(0)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()

## 7. Forward pass

A forward pass computes feature maps at **all convolutional layers**.

In [ ]:
with torch.no_grad():
    embedding = model(x).squeeze().numpy()

print("Embedding shape:", embedding.shape)

## 8. How feature maps evolve (conceptual guide)

### conv1 (7×7 kernel)
- Edges
- Color gradients
- Intensity changes

### layer1 (3×3, stride 1)
- Low‑level textures
- Repeated micro‑patterns

### layer2 (3×3)
- Canopy clusters
- Mid‑level parts

### layer3 (3×3)
- Spatial layout
- Structural arrangement

### layer4 (3×3)
- High‑level semantics
- Class‑relevant abstractions


## 9. Embedding DataFrame & CSV export

The final embedding summarizes the **strength of 512 learned concepts**.

In [ ]:
df_embedding = pd.DataFrame(
    [embedding],
    columns=[f"emb_{i}" for i in range(len(embedding))]
)
df_embedding.insert(0, "image_id", img_path.name)
df_embedding

In [ ]:
df_embedding.to_csv("image_embedding.csv", index=False)
print("Embedding saved to image_embedding.csv")